In [4]:
from utils.plot_utils import *
from utils.db_utils import *
from utils.orthanc_utils import *
import re

In [5]:
db = TinyDB(DB_PATH)
rows = []

# Iterate over all studies in the database
for study in db:
    series = study.get("series", [])

    # Construct a summary row per study
    rows.append({
        "patient_id": study.get("patient_id"),
        "patient_sex": study.get("patient_sex"),
        "age": study.get("patient_age"),
        "n_series": len(series),  # Number of series in the study
        "sax_processed": any(s.get("sax_processed", False) for s in series),  # True if any series processed
        "roundel_processed": any(s.get("roundel_processed", False) for s in series),  # True if any series roundel processed
    })

In [6]:

db = TinyDB(DB_PATH)
StudyQuery = Query()
existing_ids = [s["orthanc_study_id"] for s in db]

studies = fetch_studies()
for study_info in studies:
    # if study_info["ID"] in existing_ids:
    #     continue
    
    study = Study(study_info)
    mri_sorter = MRI_Sorter(study)
    series_label_df = mri_sorter.sort_df
    series_list = fetch_series_for_study(study.orthanc_id)

    for series_info in series_list:
        series = Series(series_info)
        if series.series_label is None: 
            series_id = series_info['ID']
            if series_id in series_label_df.index:
               series.series_label = series_label_df.loc[series_id].values[0]
            else:
               series.series_label = 'non-image'

        if not series.sax_processed: 
            # add segmentation
            pass
        if series.sax_processed and not series.roundel_processed: 
            # add roundel
            pass
        study.series_list.append(series)
    db.upsert(study.to_dict(), StudyQuery.study_uid == study.uid)
db.close()

In [3]:
series_label_df

,Description,N,Dimension,Flow,Cine,Stack,Plane,Type,Slices,Frames,Group
568c4d7a-91dfca3f-137b2c3d-cb40ff81-e5b3d465,TFE CINE FB ROOT,160,2,0,0,1,NA,Cine Stack,16,10,Cine Stack: 0
64bc5d87-7942777d-733f79d6-9dfe2313-87b0f28b,sQF font70,60,2,1,1,0,Axial,2D Flow,1,60,2D Flow: 0
7382bd26-cfb9f7cc-e9e98f70-b29ca403-1c475c84,sQF LPA 200,60,2,1,1,0,NA,2D Flow,1,60,2D Flow: 1
97b8b480-b5c32479-e1c6ee11-f022d1c5-b39ec1b2,sQF AVV 135,72,2,1,1,0,NA,2D Flow,1,72,2D Flow: 2
a17d1040-703464a5-d38b431e-0e6850ab-d5cd0989,SURVEY,90,2,0,0,0,Sagittal,Cine Single,1,30,Cine Single: 1
a489db04-36ca8d48-f123c795-b8135caf-0ce07d68,PSIR_TFE_BH SENSE,15,2,0,0,1,NA,Cine Stack,5,3,Cine Stack: 1
b37ff2bd-3f44707b-f75a1abb-dbda2589-f8bfa8c2,T1 EPI,34,2,0,0,1,Axial,Static Stack,34,1,Static Stack: 1
b86a586b-7e3bdb8e-7a7eb6bb-16a9837d-010455c8,sBTFE_4CH,360,2,0,0,1,NA,Cine Stack,12,30,Cine Stack: 3
ba3d2c93-8952b56b-083cc6ca-d015cf1f-77a073bf,sQF AO 200,60,2,1,1,0,NA,2D Flow,1,60,2D Flow: 3
ba697ee9-66b2205d-795eefb4-2d61428c-6696453e,sQF RPA 150,60,2,1,1,0,NA,2D Flow,1,60,2D Flow: 4


In [ ]:
# def extract_series_df(series_info):
#     instance_list = fetch_instances_for_series(series_info['ID'])

#     records = [
#         {
#             "SeriesID": series_info['ID'],
#             "ID": inst['ID'],
#             "ImageOrientationPatient": tuple(inst['MainDicomTags']['ImageOrientationPatient']),
#             "ImagePositionPatient": tuple(inst['MainDicomTags']['ImagePositionPatient']),
#             "InstanceNumber": inst['MainDicomTags']['InstanceNumber']
#         }
#         for inst in instance_list
#     ]
#     df = pd.DataFrame(records)
#     return df.set_index('SeriesID').sort_values('InstanceNumber')


# # Fetch all series and concatenate together
# study_df = pd.concat([extract_series_df(s) for s in fetch_series_for_study(study.orthanc_id)])

In [ ]:
# def extract_series_df(series_info):
#     '''Reading DICOMS'''
#     instance_list = fetch_instances_for_series(series_info['ID'])
#     ds = fetch_dicom(instance_list[0]['ID'])
#     records = []
#     if "PixelData" in ds and "ImageOrientationPatient" in ds:
#         for instance in instance_list:
#             record = {
#                         "SeriesUID": series_info['ID'],
#                         "ID": instance['ID'],
#                         "ImageOrientationPatient": tuple(float(x) for x in instance['MainDicomTags']['ImageOrientationPatient'].split('\\')),
#                         "ImagePositionPatient": tuple(float(x) for x in instance['MainDicomTags']['ImagePositionPatient'].split('\\')),
#                         "PixelSpacing": getattr(ds, "PixelSpacing")[0],
#                         "SliceThickness": getattr(ds, "SpacingBetweenSlices", getattr(ds, "SliceThickness")),
#                         "Dimension": int(getattr(ds, "MRAcquisitionType", np.nan).replace('D','')),
#                         "InstanceNumber": int(instance['MainDicomTags']['InstanceNumber'])
#                     }
#             records.append(record)
#         df = pd.DataFrame(records)
#         return df.set_index('SeriesID').sort_values('InstanceNumber')


# # Fetch all series and concatenate together
# study_df = pd.concat([extract_series_df(s) for s in fetch_series_for_study(study.orthanc_id)])

In [ ]:



class MRI_Sorter:
    def __init__(self, study):
        self.dicom_info = pd.concat([self.extract_series_df(s) for s in fetch_series_for_study(study.orthanc_id)])

        dicom_info_3D = self.dicom_info.loc[self.dicom_info['Dimension'] == 3]
        sort_dict_3D = self.sort_3D(dicom_info_3D)

        dicom_info_2D = self.dicom_info.loc[self.dicom_info['Dimension'] == 2]
        sort_dict_2D = self.sort_2D(dicom_info_2D) 

        sort_dict = {**sort_dict_2D, **sort_dict_3D}
        sort_df = pd.DataFrame(sort_dict).T
        # sort_df.index.name = 'SeriesUID'
        # sort_df = sort_df.reset_index()
        # sort_df['patient'] = patient
        self.sort_df = sort_df

    def orientation_id(self, iop):
        """Get ImageOrientationPatient of dicom object.

        Returns one of
            Transverse
            Coronal
            Sagittal
            NA  (if ImageOrientationPatient not available)
        """
        iop_rounded = [round(x,1) for x in iop]
        plane_cross = np.cross(iop_rounded[0:3], iop_rounded[3:6])
        plane = [abs(x) for x in plane_cross]

        if plane[0] == 1:
            return 'Sagittal'
        elif plane[1] == 1:
            return 'Coronal'
        elif plane[2] == 1:
            return 'Axial'
        else:
            return 'NA'
        

    def update_sort_dict(self, sort_dict, dimension, all_series_list, series_type, flow_flag, cine_flag, stack_flag):
        for group_tag, all_series_df in enumerate(all_series_list):
            for series_uid, series_df in all_series_df.groupby('SeriesUID'):
                sort_dict[series_uid].update({
                    'Dimension': dimension,
                    'Flow': flow_flag,
                    'Cine': cine_flag,
                    'Stack':stack_flag,
                    'ImageOrientationPatient': self.orientation_id(series_df.iloc[0].ImageOrientationPatient),
                    'Type': series_type,
                    'Slices': series_df['SliceLocation'].nunique(),
                    'Frames': int(len(series_df) / series_df['SliceLocation'].nunique()),
                    'Group': f'{series_type}: {group_tag}'
                })
        return sort_dict

    def extract_series_df(self, series_info):
        instance_list = fetch_instances_for_series(series_info['ID'])
        if not instance_list:
            return pd.DataFrame()

        # read first DICOM for shared series-level fields and RR
        ds0 = fetch_dicom(instance_list[0]['ID'])
        if not ("PixelData" in ds0 and "ImageOrientationPatient" in ds0):
            return pd.DataFrame()

        image_shape = (ds0.Rows, ds0.Columns)
        series_fields = {
            "SeriesUID": series_info['ID'],
            "SeriesDescription": getattr(ds0, "SeriesDescription", None),
            "PixelSpacing": getattr(ds0, "PixelSpacing")[0],
            "SliceThickness": getattr(ds0, "SpacingBetweenSlices", getattr(ds0, "SliceThickness", np.nan)),
            "Dimension": int(getattr(ds0, "MRAcquisitionType", "0").replace("D", "")) if hasattr(ds0, "MRAcquisitionType") else np.nan,
            "N_timesteps": int(getattr(ds0, "CardiacNumberOfImages", 0)),
            "ImageShape": image_shape,
            "venc": 0,
        }

        # compute RR only from first DICOM
        try:
            rr_ni = float(ds0.NominalInterval)
        except Exception:
            rr_ni = 0.0
        try:
            rr_hr = 60000.0 / float(ds0.HeartRate)
        except Exception:
            rr_hr = 0.0
        rr = max(rr_ni, rr_hr)
        series_fields["rr"] = rr if 100 < rr < 3000 else 0.0

        # build records using only MainDicomTags
        records = [
            {
                **series_fields,
                "ID": inst['ID'],
                "ImageOrientationPatient": tuple(float(x) for x in inst['MainDicomTags']['ImageOrientationPatient'].split('\\')),
                "ImagePositionPatient": tuple(float(x) for x in inst['MainDicomTags']['ImagePositionPatient'].split('\\')),
                "SliceLocation": float(inst['MainDicomTags']['ImagePositionPatient'].split('\\')[2]),
                "InstanceNumber": int(inst['MainDicomTags']['InstanceNumber']),
                "TriggerTime": int(inst['MainDicomTags']['InstanceNumber']),
            }
            for inst in instance_list
        ]

        return pd.DataFrame(records).sort_values("InstanceNumber")

    # def extract_series_df(self, series_info):
    #     instance_list = fetch_instances_for_series(series_info['ID'])
    #     if not instance_list:
    #         return pd.DataFrame()

    #     ds0 = fetch_dicom(instance_list[0]['ID'])
    #     if not ("PixelData" in ds0 and "ImageOrientationPatient" in ds0):
    #         return pd.DataFrame()

    #     image_shape = (ds0.Rows, ds0.Columns)

    #     series_fields = {
    #         "SeriesUID": series_info['ID'],
    #         "SeriesDescription": getattr(ds0, "SeriesDescription", None),
    #         "PixelSpacing": getattr(ds0, "PixelSpacing")[0],
    #         "SliceThickness": getattr(ds0, "SpacingBetweenSlices", getattr(ds0, "SliceThickness", np.nan)),
    #         "Dimension": int(getattr(ds0, "MRAcquisitionType", "0").replace("D", "")) if hasattr(ds0, "MRAcquisitionType") else np.nan,
    #         "N_timesteps": int(getattr(ds0, "CardiacNumberOfImages", 0)),
    #         "ImageShape": image_shape,
    #         "venc": 0
    #     }

    #     # fetch all dicoms once
    #     dcms = {inst['ID']: fetch_dicom(inst['ID']) for inst in instance_list}

        

    #     records = [
    #         {
    #             **series_fields,
    #             "ID": inst['ID'],
    #             "ImageOrientationPatient": tuple(map(float, inst['MainDicomTags']['ImageOrientationPatient'].split('\\'))),
    #             "ImagePositionPatient": tuple(map(float, inst['MainDicomTags']['ImagePositionPatient'].split('\\'))),
    #             "SliceLocation": float(inst['MainDicomTags']['ImagePositionPatient'].split('\\')[2]),
    #             "InstanceNumber": int(inst['MainDicomTags']['InstanceNumber']),
    #             "TriggerTime": getattr(dcms[inst['ID']], "TriggerTime", np.nan),
    #             "rr": compute_rr(dcms[inst['ID']])
    #         }
    #         for inst in instance_list
    #     ]   

    #     return pd.DataFrame(records).sort_values("InstanceNumber")
    
    def sort_cine_stacks(self, cine_non_flow_list):
        """
        Sort the 2D DICOMs into cine stacks and update the sort dictionary.
        """
        cine_non_flow_df = pd.concat(cine_non_flow_list)
        cine_stack_list = []
        
        # Process each unique orientation
        for unique_or in cine_non_flow_df['ImageOrientationPatient'].dropna().unique():
            # Filter by orientation and sort by pixel spacing, thickness, and timesteps
            unique_df = cine_non_flow_df[cine_non_flow_df['ImageOrientationPatient'] == unique_or].reset_index()
            unique_df = unique_df.set_index(['PixelSpacing', 'SliceThickness', 'N_timesteps']).sort_index()

            for unique_idx, stack_df in unique_df.groupby(level=[0, 1, 2]):
                # Process each unique image shape
                for unique_imshape, shape_group in stack_df.groupby('ImageShape'):
                    shape_group = shape_group.reset_index()
                    
                    if len(shape_group) > 1:  # Ensure there are multiple entries
                        separated = False
                        
                        # Process each unique series
                        for uni_series, series_df in shape_group.groupby('SeriesUID'):
                            if len(series_df) > 50:  # Large series: separate based on SliceLocation
                                if series_df['SliceLocation'].nunique() > 1:
                                    cine_stack_list.append(series_df)
                                    separated = True
                        
                        # If not separated, add the entire group if it has multiple slice locations
                        if not separated and shape_group['SliceLocation'].nunique() > 1:
                            cine_stack_list.append(shape_group)
        
        if cine_stack_list:
            cine_stack_ = pd.concat(cine_stack_list)['SeriesUID'].unique()
            cine_single_list = [df for df in cine_non_flow_list if df['SeriesUID'].iloc[0] not in cine_stack_SeriesUIDs] # get the single cines
        else:
            cine_single_list = cine_non_flow_list
        return cine_stack_list, cine_single_list


    def get_venc(self, cine_single_list):
        for series_df in cine_single_list:
            max_venc = 0

            idxs = (0, len(series_df)//2, len(series_df)-1)
            for i in idxs:
                ds = fetch_dicom(series_df.iloc[i]['ID'])
                manufacturer = (getattr(ds, 'Manufacturer', '') or '').lower()

                venc = 0
                if 'siemens' in manufacturer:
                    val = getattr(ds.get((0x0051, 0x1014)), 'value', None)
                    if val:
                        nums = re.findall(r'\d+', str(val))
                        if nums:
                            venc = float(max(map(int, nums)))
                    if not venc:
                        val = getattr(ds.get((0x0018, 0x0024)), 'value', '')
                        m = re.search(r'v(\d+)in', str(val))
                        if m:
                            venc = float(m.group(1))

                elif 'ge' in manufacturer:
                    val = getattr(ds.get((0x0019, 0x10cc)), 'value', None)
                    if val:
                        venc = val / 10

                elif 'philips' in manufacturer:
                    seq = getattr(ds, 'RealWorldValueMappingSequence', None)
                    if seq:
                        intercept = getattr(seq[0], 'RealWorldValueIntercept', None)
                        if intercept is not None:
                            venc = abs(intercept)
                    if not venc:
                        intercept = getattr(ds, 'RescaleIntercept', None)
                        if intercept is not None:
                            venc = abs(intercept)

                max_venc = max(max_venc, venc)

            series_df['venc'] = max_venc

        return cine_single_list
   
    def sort_2D(self, dicom_info_2D):

        sort_dict_2D = {}

        for series_uid, series_df in dicom_info_2D.groupby('SeriesUID'):
            # Initialize the dictionary for each series
            sort_dict_2D[series_uid] = {
                'Description': series_df.iloc[0]['SeriesDescription'],
                'N': len(series_df),
                'Dimension': 2
            }

        # Initialize lists to store results
        series_df_list = []

        # Process unique ImageOrientationPatients
        for unique_or in dicom_info_2D['ImageOrientationPatient'].dropna().unique():
            # Filter and sort DataFrame
            unique_df = dicom_info_2D[dicom_info_2D['ImageOrientationPatient'] == unique_or]
            unique_df = unique_df.set_index(['PixelSpacing', 'SliceThickness']).sort_index()

            # Group by unique indices (PixelSpacing, SliceThickness)
            for unique_idx, series_df in unique_df.groupby(level=[0, 1]):
                # Group by image shape
                for unique_imshape, shape_group in series_df.groupby('ImageShape'):
                    # Filter for series with more than one entry
                    for uni_series, series_df in shape_group.groupby('SeriesUID'):
                        if len(series_df) > 0:
                            series_df_list.append(series_df)

            stack_df_list = [df for df in series_df_list if df['SliceLocation'].nunique() > 1]
            single_df_list = [df for df in series_df_list if df['SliceLocation'].nunique() == 1]

            static_stack_list = [
                        df
                        for df in stack_df_list
                        if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() == 1
                    ]

            cine_stack_list = [
                        df
                        for df in stack_df_list
                        if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() > 1
                    ]


            static_single_list = [
                        df
                        for df in single_df_list
                        if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() == 1
                    ]

            cine_single_list = [
                        df
                        for df in single_df_list
                        if df.loc[df['SliceLocation'] == df['SliceLocation'].unique()[0]]['InstanceNumber'].nunique() > 1 
                    ]

            cine_single_list = self.get_venc(cine_single_list)

            cine_2d_flow_list = [df for df in cine_single_list if (df['venc'] > 0).any()]
            cine_single_list = [df for df in cine_single_list if not (df['venc'] > 0).any()]



        sort_dict_2D = self.update_sort_dict(sort_dict_2D, 2, static_single_list, 'Static Single', flow_flag=0, cine_flag=0, stack_flag = 0)
        sort_dict_2D = self.update_sort_dict(sort_dict_2D, 2, static_stack_list, 'Static Stack', flow_flag=0, cine_flag=0, stack_flag = 1)
        
        sort_dict_2D = self.update_sort_dict(sort_dict_2D, 2, cine_single_list, 'Cine Single', flow_flag=0, cine_flag=0, stack_flag = 0)
        sort_dict_2D = self.update_sort_dict(sort_dict_2D, 2, cine_stack_list, 'Cine Stack', flow_flag=0, cine_flag=0, stack_flag = 1)

        sort_dict_2D = self.update_sort_dict(sort_dict_2D, 2, cine_2d_flow_list, '2D Flow', flow_flag=1, cine_flag=1, stack_flag = 0)

        return sort_dict_2D
    
    def sort_3D(self, dicom_info_3D):
        '''
        sort the 3D dicoms
        '''
        sort_dict_3D = {}
        for series_uid, series_df in dicom_info_3D.groupby('SeriesUID'):
            sort_dict_3D[series_uid] = {
                'Description': series_df.iloc[0]['SeriesDescription'],
                'N': len(series_df),
                'Dimension': 3
            }

        # Initialize lists to store results
        stack_df_list = []

        # Process unique orientations
        for unique_or in dicom_info_3D['ImageOrientationPatient'].dropna().unique():
            # Filter and sort DataFrame
            unique_df = dicom_info_3D[dicom_info_3D['ImageOrientationPatient'] == unique_or]
            unique_df = unique_df.set_index(['PixelSpacing', 'SliceThickness']).sort_index()

            # Group by unique indices (pixelspacing, thickness)
            for unique_idx, stack_df in unique_df.groupby(level=[0, 1]):
                # Group by image shape
                for unique_imshape, shape_group in stack_df.groupby('ImageShape'):
                    if len(shape_group) > 1:
                        stack_df_list.append(shape_group)

        # Classify single-phase and multi-phase datasets
        stack_df_list = self.get_venc(stack_df_list)

        flow_4d = [df for df in stack_df_list if (df['venc'] > 0).any()]
        single_phase_list = [df for df in stack_df_list if df['SliceLocation'].nunique() == len(df) and not (df['venc'] != 0).any()]
        multi_phase_list = [df for df in stack_df_list if df['SliceLocation'].nunique() != len(df) and not (df['venc'] != 0).any()]

        single_mra = [df for df in single_phase_list if df['rr'].max() == 0]
        single_whole_heart = [df for df in single_phase_list if df['rr'].max() > 0]
        multi_mra = [df for df in multi_phase_list if df['rr'].max() == 0]
        cine_4d = [df for df in multi_phase_list if (df['rr'].max() > 0) & (df['TriggerTime'].nunique() > 1)]

        # Update the sort_dict_3D 
        sort_dict_3D = self.update_sort_dict(sort_dict_3D, 3, single_mra, 'Single MRA', flow_flag=0, cine_flag=0, stack_flag = 'NA')
        sort_dict_3D = self.update_sort_dict(sort_dict_3D, 3, single_whole_heart, 'Single Whole-Heart', flow_flag=0, cine_flag=0, stack_flag = 'NA')
        sort_dict_3D = self.update_sort_dict(sort_dict_3D, 3, multi_mra, 'Multi MRA', flow_flag=0, cine_flag=0, stack_flag = 'NA')
        sort_dict_3D = self.update_sort_dict(sort_dict_3D, 3, flow_4d, '4D Flow', flow_flag=1, cine_flag=1, stack_flag = 'NA')
        sort_dict_3D = self.update_sort_dict(sort_dict_3D, 3, cine_4d, '4D Cine', flow_flag=0, cine_flag=1, stack_flag = 'NA')
        return sort_dict_3D

In [ ]:
mri_sorter = MRI_Sorter(study)